# Scalable Entity Resolution for Property Graphs with AutoGraft

When building large-scale GraphRAG applications using `PropertyGraphIndex` with `Neo4jPropertyGraphStore`, entity resolution (deduplication) becomes a massive bottleneck. 

Naively using LLMs to deduplicate every incoming entity against your entire graph creates an $O(N \times M)$ scaling nightmare that burns API credits and halts ingestion.

This notebook demonstrates how to use the open-source **AutoGraft** middleware. AutoGraft intercepts nodes extracted by LlamaIndex and pushes the deduplication logic directly to Neo4j's native B-Tree and HNSW Vector indexes, reducing the time complexity to $O(N \log M)$ and eliminating up to 99% of unnecessary LLM calls.

## 1. Setup Environment
First, install the required dependencies:

In [ ]:
!pip install llama-index llama-index-graph-stores-neo4j autograft

## 2. Initialize Neo4j and LlamaIndex Extractors
We start by connecting to our Neo4j database and setting up a standard `PropertyGraphIndex` extraction pipeline.

In [ ]:
import os
from llama_index.core import PropertyGraphIndex
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core.indices.property_graph import SimpleLLMPathExtractor

os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "password"
os.environ["OPENAI_API_KEY"] = "sk-..."

# Initialize the standard Neo4j Graph Store
graph_store = Neo4jPropertyGraphStore(
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    url=os.environ["NEO4J_URI"],
)

# Standard extractor that generates raw entities
kg_extractor = SimpleLLMPathExtractor(
    max_paths_per_chunk=10,
    num_workers=4,
)

## 3. The AutoGraft Magic: Injecting the Middleware
Instead of passing the raw extractor directly to the index, we wrap it in `AutoGraftNeo4jMiddleware`. 

The middleware acts as a gatekeeper:
1. **Deterministic Layer:** Checks Neo4j B-Tree indexes for exact matches.
2. **Semantic Layer:** If no exact match, queries Neo4j's HNSW vector index for semantic proximity.
3. **LLM Layer:** Only triggers an LLM resolution call if the semantic score is highly ambiguous.


In [ ]:
from autograft.integrations.llamaindex import AutoGraftNeo4jMiddleware

# Wrap the extractor with the scalable AutoGraft middleware
autograft_extractor = AutoGraftNeo4jMiddleware(
    extractor=kg_extractor,
    neo4j_url=os.environ["NEO4J_URI"],
    neo4j_user=os.environ["NEO4J_USERNAME"],
    neo4j_password=os.environ["NEO4J_PASSWORD"],
    similarity_threshold=0.92,  # Below 0.92 = New node
    uncertainty_threshold=0.96,  # Between 0.92 and 0.96 = LLM call. Above 0.96 = Automatic Merge
)

# Now pass the wrapped extractor to the PropertyGraphIndex
index = PropertyGraphIndex.from_documents(
    documents,
    kg_extractors=[autograft_extractor],
    property_graph_store=graph_store,
    show_progress=True,
)

## 4. Querying the Deduplicated Graph
Because AutoGraft resolves entities in real-time before they are written, your Neo4j graph remains perfectly clean and free of duplicates without exploding your API costs.

You can now query the `PropertyGraphIndex` natively:

In [ ]:
retriever = index.as_retriever(
    include_text=True,
)

nodes = retriever.retrieve("What are the key products created by Apple?")
for node in nodes:
    print(node.text)